In [0]:
%pip install kaggle

In [0]:
%restart_python

In [0]:
%run ../config/config_reader

In [0]:
import os
import subprocess
import tempfile
import shutil
import hashlib

from pathlib import Path

# Load project configuration
config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

if not os.path.exists(config_path):
    raise FileNotFoundError(
        f"Configuration file not found: {config_path}"
    )

config = load_config(config_path)

print("Configuration loaded successfully!")

In [0]:
# Retrieve Kaggle API token
KAGGLE_API_TOKEN = dbutils.secrets.get(
    scope="kaggle",
    key="kaggle_data_extract"
)

if not KAGGLE_API_TOKEN:
    raise ValueError("Kaggle API token is missing.")

# Configure Kaggle authentication
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print("Kaggle API token retrieved successfully.")

In [0]:
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize Kaggle API
api = KaggleApi()
api.authenticate()

# Create temporary download directory
download_dir = tempfile.mkdtemp(prefix="zomato_kaggle_")

# Read dataset ID from config
dataset_id = config["source"]["dataset_id"]

# Download and extract dataset
api.dataset_download_files(
    dataset=dataset_id,
    path=download_dir,
    unzip=True
)

print("Dataset downloaded and extracted successfully.")

In [0]:
downloaded_files = [
    file for file in Path(download_dir).rglob("*")
    if file.is_file()
]

print(f"Total files extracted: {len(downloaded_files)}")

for file in downloaded_files:
    print(
        f"File: {file.name} | "
        f"Size: {file.stat().st_size:,} bytes | "
        f"Path: {file}"
    )

csv_files = [
    file for file in downloaded_files
    if file.suffix.lower() == ".csv"
]

print("\nCSV files found:")

for file in csv_files:
    print(file.name)

In [0]:
# Read target configuration
target_config = config["target"]

# Construct landing path dynamically
landing_path = (
    f"/Volumes/{target_config['catalog']}/"
    f"{target_config['bronze_schema']}/"
    f"{target_config['volume']}/"
    f"{target_config['landing_directory']}"
)

# Create landing directory
os.makedirs(landing_path, exist_ok=True)

# Source and destination
source_file = csv_files[0]

destination_file = os.path.join(
    landing_path,
    source_file.name
)

# Copy file to Unity Catalog Volume
shutil.copy2(
    str(source_file),
    destination_file
)

print(f"File landed successfully: {destination_file}")